In [72]:
import sys
from pathlib import Path
# Add src directory to path to import utils package
sys.path.insert(0, str(Path('..') / 'src'))

from utils.io import (
    list_containers,
    explore_lif_container,
    load_lif_image,
    calculate_rescale_factor,
    ensure_output_dir,
    load_precomputed_results_if_available,
)
from utils.segmentation import (
    predict_nuclei_labels,
    simulate_fluo_from_bf,
    generate_rough_root_3d,
    fill_root_3d,
    smooth_outer_root_surface_3d
)
from utils.inference import predict_tiled_unet

from utils.feature_extraction import calculate_distance_to_root_surface

import tifffile
import napari
import torch

In [73]:
# Copy the path where your .lif containers are stored, you can use absolute or relative paths to point at other disk locations
RAW_DATA_DIRECTORY = r"../raw_data"

# Point to the local PanSeg model UNet3D weights and config files
# You can choose from lightsheet_3D_unet_root_ds1x, lightsheet_3D_unet_root_ds2x, lightsheet_3D_unet_root_ds3x, confocal_3D_unet_sa_meristem_cells & generic_confocal_3D_unet
MODEL_DIR = "../plantseg_models/lightsheet_3D_unet_root_ds3x"

# Channel index used for CellposeSAM-based 3D nuclei segmentation
NUCLEI_CHANNEL = 2

# FRET-based biosensor system (nlsABACUS2-400)
# edCerulean_CTRL: excitation of edCerulean, emission of edCerulean (donor) 
# edCitrine_FRET: excitation of edCerulean (donor), emission of edCitrine (acceptor) – FRET
# edCitrine_CTRL: excitation of edCitrine, emission of edCitrine (acceptor) - used for nuclei segmentation
MARKERS = (("edCerulean_CTRL", 0), ("edCitrine_FRET", 1), ("edCitrine_CTRL", 2), ("brightfield", 3))

# Mark the position of the .lif container you want to open in your raw data folder (first = 0, second = 1, third = 2)
LIF_CONTAINER_INDEX = 1 

# Mark the position of the image inside the .lif container you want to open (first = 0, second = 1, third = 2)
LIF_IMAGE_INDEX = 0

In [74]:
# Iterate through the .lif container files in the directory
lif_containers = list_containers(RAW_DATA_DIRECTORY, file_format="lif")

lif_containers

['..\\raw_data\\20260129_ABACUS timepoints_mock 3h.lif',
 '..\\raw_data\\20260129_ABACUS timepoints_sorb 3h.lif',
 '..\\raw_data\\20260204_Splitplate_mock_6h.lif',
 '..\\raw_data\\20260204_Splitplate_sorb_6h.lif']

In [75]:
# Explore different .lif files (0 defines the first file in the directory)
lif_path = lif_containers[LIF_CONTAINER_INDEX]

# Explore the contents of a single .lif container
nr_imgs, lif_container_id = explore_lif_container(file_path=lif_path, display=True)

# Load a single image from a .lif container
lif_image, lif_image_name, xml_metadata = load_lif_image(file_path=lif_path, image_index=LIF_IMAGE_INDEX)

Image name: Col0 1, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (125, 4, 256, 1024)
Image name: Col0 2, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (121, 4, 256, 1024)
Image name: Col0 3, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (149, 4, 256, 1024)
Image name: Col0 4, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (126, 4, 256, 1024)
Image name: Col0 5, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (120, 4, 256, 1024)
Image name: Col0 6, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (114, 4, 256, 1024)
Image name: Col0 7, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (174, 4, 256, 1024)
Image name: Col0 8, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (111, 4, 256, 1024)
Image name: Col0 9, Dimensions: ('Z', 'C', 'Y', 'X'), Array Shape: (130, 4, 256, 1024)


In [76]:
# Initialize Napari Viewer and display all input channels
viewer = napari.Viewer(ndisplay=3)
viewer.add_image(lif_image, 
                channel_axis=0,
                colormap=['cyan', 'yellow', 'magenta', 'inferno'],
                name=[tuple[0] for tuple in MARKERS] #['edCerulean_CTRL','edCitrine_FRET','edCitrine_CTRL','brightfield']
                )

[<Image layer 'edCerulean_CTRL' at 0x1d49f8dc3d0>,
 <Image layer 'edCitrine_FRET' at 0x1d49f907ca0>,
 <Image layer 'edCitrine_CTRL' at 0x1d49f965d50>,
 <Image layer 'brightfield' at 0x1d49f99ad70>]

In [77]:
# Simulate fluorescently labelled cell walls from brightfield input image
sim_fluo_cell_walls = simulate_fluo_from_bf(lif_image, MARKERS)

# Add simulated fluorescently labelled plant cell boundaries to Napari viewer
viewer.add_image(sim_fluo_cell_walls,
                name="PanSeg_UNet3D_input",
                colormap="gray",
                blending="additive")


<Image layer 'PanSeg_UNet3D_input' at 0x1d46824d0c0>

In [78]:
# Ensure output directory for this container's nuclei labels
nuclei_labels_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="nuclei_labels")
print(f"Nuclei labels directory: {nuclei_labels_dir}")

# Calculate anisotropy CellposeSAM parameter to rescale across the Z-axis (ratio of Z-resolution to XY-resolution)
rescale_factor = calculate_rescale_factor(xml_metadata, display=True)

# Load precomputed labels when available; otherwise predict and store them
nuclei_labels = load_precomputed_results_if_available(nuclei_labels_dir, lif_image_name, results_type="nuclei_labels")

if nuclei_labels is not None:
    print(f"Predictions already calculated for: {lif_image_name} ...loading")
    # Add the prediction to the napari viewer
    viewer.add_labels(nuclei_labels)
    
else:
    # Predict nuclei labels using CellposeSAM using anisotropy correction
    nuclei_labels = predict_nuclei_labels(lif_image, rescale_factor, NUCLEI_CHANNEL, visualize=True)
    # Create path for nuclei labels (used only when saving a newly computed prediction)
    nuclei_labels_path = nuclei_labels_dir / f"{lif_image_name}_nuclei_labels.tif"
    # Save the prediction
    tifffile.imwrite(nuclei_labels_path, nuclei_labels)



Nuclei labels directory: ..\raw_data\nuclei_labels\20260129_ABACUS timepoints_sorb 3h
x_um: 0.7568359375, y_um: 0.75461640625, z_um: 0.399525
Rescale factor: 0.5286637076611433
2026-04-19 03:41:01,850 [INFO] resizing 3D image with anisotropy=0.5286637076611433
2026-04-19 03:41:01,983 [INFO] running YX: 66 planes of size (256, 1024)
2026-04-19 03:41:33,438 [INFO] 100%|##########| 66/66 [00:31<00:00,  2.10it/s]
2026-04-19 03:41:33,622 [INFO] running ZY: 256 planes of size (66, 1024)
2026-04-19 03:42:38,214 [INFO] 100%|##########| 256/256 [01:04<00:00,  3.96it/s]
2026-04-19 03:42:38,411 [INFO] running ZX: 1024 planes of size (66, 256)
2026-04-19 03:44:09,489 [INFO] 100%|##########| 256/256 [01:31<00:00,  2.81it/s]
2026-04-19 03:44:09,958 [INFO] network run in 188.11s
2026-04-19 03:44:15,519 [INFO] masks created in 3.97s


In [79]:
# Ensure output directory for this container's 3D root mask
root_mask_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="root_mask")
print(f"3D Root Mask directory: {root_mask_dir}")

# Load precomputed root mask when available; otherwise generate and store it
smooth_root_3d = load_precomputed_results_if_available(root_mask_dir, lif_image_name, results_type="root_mask")

if smooth_root_3d is not None:
    print(f"3D root mask already calculated for: {lif_image_name} ...loading")
    # Add the precomputed mask to the napari viewer
    viewer.add_image(smooth_root_3d, name="smooth_root_3d", colormap="green", blending="additive", opacity=0.5)

else:
    # Predict root cell boundary probability maps using a pre-trained UNet3D model
    root_pmaps = predict_tiled_unet(
        raw=sim_fluo_cell_walls,
        input_layout="ZYX",
        model_dir=MODEL_DIR,
        patch=(80, 160, 160),
        patch_halo=(8, 16, 16),
        stride_ratio=0.75,
        batch_size=1,
        device="cuda" if torch.cuda.is_available() else "cpu",
        use_amp=True,
    )

    # root_pmaps: (C_out, Z, Y, X)
    viewer.add_image(root_pmaps[0], name="root_unet_pmap", colormap="viridis", blending="additive")

    # Generate a rough 3D root mask
    filled_3d_closed = generate_rough_root_3d(root_pmaps, nuclei_labels, probability_threshold=0.9, visualize=True)
    # Fill internal gaps inside rough 3D root mask
    filled_root_3d = fill_root_3d(filled_3d_closed, occupancy_threshold=0.9, slice_aware_filling=True, visualize=True)
    # Smooth root outer surface to remove small protrusions
    smooth_root_3d = smooth_outer_root_surface_3d(filled_root_3d, erosion=5, smoothing=3, visualize=True)
    # Create path for root mask (used only when saving a newly computed prediction)
    root_mask_path = root_mask_dir / f"{lif_image_name}_root_mask.tif"
    # Save the processed mask
    tifffile.imwrite(root_mask_path, smooth_root_3d)

3D Root Mask directory: ..\raw_data\root_mask\20260129_ABACUS timepoints_sorb 3h


c:\Users\adiezsanchez\githubrepos\saramorg_fret_nroot\notebooks\..\src\utils\inference.py:163: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(weights_path,

# #TODO: VERY IMPORTANT: CONSIDER REMOVE PADDING FROM BOTTOM OF THE STACK BEFORE CALCULATING DISTANCE MAP (T)

In [80]:
# Ensure output directory for this container's nuclei depth_map
depth_map_dir = ensure_output_dir(RAW_DATA_DIRECTORY, lif_container_id, results_type="depth_map")
print(f"Nuclei depth map directory: {depth_map_dir}")

# Load precomputed depth map when available; otherwise generate and store it
nuclei_depth_map = load_precomputed_results_if_available(depth_map_dir, lif_image_name, results_type="depth_map")

if nuclei_depth_map is not None:
    print(f"Nuclei depth map already calculated for: {lif_image_name} ...loading")
    # Add the precomputed map to Napari
    viewer.add_image(nuclei_depth_map, name="nuclei_depth_normalized", colormap="viridis", blending="additive")

else:
    # Calculate distance from each nuclei centroid to the root surface
    # This will be used to approximate to which tissue layer each nucleus belongs
    nuclei_depth_map = calculate_distance_to_root_surface(nuclei_labels, smooth_root_3d, visualize=True)
    # Create path for nuclei depth map (used only when saving a newly computed prediction)
    nuclei_depth_path = depth_map_dir / f"{lif_image_name}_depth_map.tif"
    # Save the calculated depth map
    tifffile.imwrite(nuclei_depth_path, nuclei_depth_map)

Nuclei depth map directory: ..\raw_data\depth_map\20260129_ABACUS timepoints_sorb 3h
